In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import os, sys, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm.notebook import tqdm

sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/spikeparam')
sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/AP_empirical_paper1/datasets/spe-1/spe1_helper_modules/')

from config import SPE1_PICKLE_ROOT, CELL_IDS, DICT_CELL_TYPE
from spikeparam.patch.fit import Spike

plt.rcParams['font.family'] = 'Helvetica Neue'

SPE1_PKL = SPE1_PICKLE_ROOT
PVC6_PKL = '/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/pvc6_pickles'

In [ ]:
FORCE_RERUN = False   # set True to refit pvc-6 and overwrite R² caches

## pvc-6 R² — refit from all_data pickles and cache

In [ ]:
def get_pvc6_r2(all_data_pkl, cache_pkl, fs=200000, force=False):
    """Fit Spike on pvc-6 all_data and cache r2_exp / r2_ramp."""
    if not force and os.path.exists(cache_pkl):
        with open(cache_pkl, 'rb') as f:
            d = pickle.load(f)
        # backfill old cache that used 'r_squared_exp' key
        if 'r_squared_exp' in d:
            d = {'r2_exp': d['r_squared_exp'], 'r2_ramp': d['r_squared_ramp']}
        return d

    with open(all_data_pkl, 'rb') as f:
        all_data = pickle.load(f)
    all_data_flat = [s for sweep in all_data for s in sweep]

    sp = Spike(thresh_amp=-10, window_length=(5., 5.), smooth_frac=.008,
               pre_inflection_ms=0.5)
    sp.fit(all_data_flat, fs, n_jobs=-1, progress=tqdm)
    sp.gen_fit(ramp=True, exp=True)

    result = {
        'r2_exp':  np.asarray(sp.r_squared_exp,  dtype=float),
        'r2_ramp': np.asarray(sp.r_squared_ramp, dtype=float),
    }
    with open(cache_pkl, 'wb') as f:
        pickle.dump(result, f)
    print(f'Cached to {os.path.basename(cache_pkl)}')
    return result


pvc6_r2 = {
    'c1': get_pvc6_r2(
        os.path.join(PVC6_PKL, 'all_data.pkl'),
        os.path.join(PVC6_PKL, '_r2_c1.pkl'),
        force=FORCE_RERUN,
    ),
    'c2': get_pvc6_r2(
        os.path.join(PVC6_PKL, 'all_data2.pkl'),
        os.path.join(PVC6_PKL, '_r2_c2.pkl'),
        force=FORCE_RERUN,
    ),
}

for cid, d in pvc6_r2.items():
    print(f'pvc-6 {cid}: n={len(d["r2_exp"])}  '
          f'median r²_exp={np.nanmedian(d["r2_exp"]):.3f}  '
          f'median r²_ramp={np.nanmedian(d["r2_ramp"]):.3f}')

## spe-1 R² — load from spike_fit_pickles

In [ ]:
_fit_dir     = os.path.join(SPE1_PKL, 'spike_fit_pickles')
_cluster_dir = os.path.join(SPE1_PKL, 'cluster_pickles')

spe1_r2 = []   # list of dicts: cell_id, cell_type, r2_exp, r2_ramp

for _cid in CELL_IDS:
    # only include cells present in the main analysis (cluster pickles)
    _cluster_p = os.path.join(_cluster_dir, f'{_cid}_cluster_df.pkl')
    if not os.path.exists(_cluster_p):
        continue

    _fit_p = os.path.join(_fit_dir, f'{_cid}_spike_fit.pkl')
    if not os.path.exists(_fit_p):
        continue

    with open(_fit_p, 'rb') as _f:
        _sp = pickle.load(_f)

    if _sp.r_squared_exp is None:
        _sp.gen_fit(ramp=True, exp=True)

    _cnum  = int(_cid.replace('c', ''))
    _ctype = DICT_CELL_TYPE.get(_cnum, 'PC')
    spe1_r2.append({
        'cell_id':   _cid,
        'cell_type': _ctype,
        'r2_exp':    np.asarray(_sp.r_squared_exp,  dtype=float),
        'r2_ramp':   np.asarray(_sp.r_squared_ramp, dtype=float),
    })

print(f'Loaded {len(spe1_r2)} spe-1 cells (cluster-pickle gated)')
for d in spe1_r2:
    print(f"  {d['cell_id']} ({d['cell_type']}): n={len(d['r2_exp'])}  "
          f"med r²_exp={np.nanmedian(d['r2_exp']):.3f}  "
          f"med r²_ramp={np.nanmedian(d['r2_ramp']):.3f}")

## R² distributions — summary + per-cell

In [ ]:
import matplotlib
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from scipy.stats import pearsonr
import pandas as pd

matplotlib.rcParams['font.family']     = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['Helvetica Neue', 'Helvetica', 'Arial']

_FS_AX  = 28
_FS_TK  = 22
_FS_SM  = 14
_LW_SP  = 2.5
_ALPHA  = 0.82

# Paper color conventions (from spe1_dataset_summary / pvc6_dataset_summary)
COL_PC   = '#CC44CC'   # magenta — putative PC
COL_IN   = '#00CCCC'   # cyan    — putative IN
COL_PVC6 = '#888888'   # gray    — pvc-6 dataset
COL_SPE1 = '#5A5A80'   # muted indigo — spe-1 combined (summary panels)

SHAPE_FEATS = ['peak_amp', 'peak_width', 'peak_sharpness', 'inflection_amp', 'inflection_time']
FEAT_NICE   = ['Peak\namp', 'Peak\nwidth', 'Sharp-\nness', 'Infl.\namp', 'Infl.\ntime']


# ── helpers ───────────────────────────────────────────────────────────────────

def _pool_spe1(r2_key):
    arrs = [d[r2_key][~np.isnan(d[r2_key])] for d in spe1_r2]
    return np.concatenate(arrs) if arrs else np.array([])

def _pool_pvc6(r2_key):
    return np.concatenate([d[r2_key][~np.isnan(d[r2_key])] for d in pvc6_r2.values()])

def _style(ax, xtick_sz=None):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for sp in ['bottom', 'left']:
        ax.spines[sp].set_linewidth(_LW_SP)
    ax.tick_params(axis='both', width=_LW_SP, labelsize=xtick_sz or _FS_TK)
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontweight('bold')

def _boxes(ax, data, pos, cols, widths=0.55):
    bp = ax.boxplot(data, positions=pos, widths=widths, patch_artist=True,
                    medianprops=dict(color='black', linewidth=2.2),
                    whiskerprops=dict(linewidth=_LW_SP * 0.9),
                    capprops=dict(linewidth=_LW_SP * 0.9),
                    showfliers=False,
                    showmeans=True,
                    meanprops=dict(marker='D', markerfacecolor='white',
                                   markeredgecolor='white', markersize=9, zorder=5))
    for p, c in zip(bp['boxes'], cols):
        p.set_facecolor(c); p.set_alpha(_ALPHA); p.set_linewidth(0)
    for w, c in zip(bp['whiskers'], [c for c in cols for _ in (0, 1)]):
        w.set_color(c); w.set_linewidth(_LW_SP * 0.9)
    for cp, c in zip(bp['caps'], [c for c in cols for _ in (0, 1)]):
        cp.set_color(c); cp.set_linewidth(_LW_SP * 0.9)

def _r2_yax(ax):
    ax.set_ylim(-0.05, 1.05)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['0', '0.25', '0.5', '0.75', '1'], fontweight='bold', fontsize=_FS_TK)
    ax.axhline(0.5, color='#aaaaaa', ls='--', lw=1.5, alpha=0.5)

def _fit_err_yax(ax):
    # log-scale fit error so pvc-6 (near 0) and spe-1 (near 0.3-0.8) are both visible
    ax.set_yscale('log')
    ax.set_ylim(5e-4, 2.5)
    ax.set_yticks([1e-3, 1e-2, 1e-1, 1.0])
    ax.set_yticklabels(['10⁻³', '10⁻²', '10⁻¹', '1'], fontweight='bold', fontsize=_FS_TK)


# ── build per-cell CV data ────────────────────────────────────────────────────
_fit_dir = os.path.join(SPE1_PKL, 'spike_fit_pickles')

cv_rows = []
for d in spe1_r2:
    cid = d['cell_id']
    with open(os.path.join(_fit_dir, f'{cid}_spike_fit.pkl'), 'rb') as _f:
        _sp = pickle.load(_f)
    df_f = _sp.df_features
    row = {'cid': cid, 'ct': d['cell_type'],
           'r2_exp':  np.nanmedian(d['r2_exp']),
           'r2_ramp': np.nanmedian(d['r2_ramp'])}
    for feat in SHAPE_FEATS:
        vals = df_f[feat].dropna().values
        mu = np.abs(np.nanmean(vals))
        row[f'cv_{feat}'] = np.nanstd(vals) / mu if mu > 1e-9 else np.nan
    cv_rows.append(row)
df_cv = pd.DataFrame(cv_rows)
df_cv['mean_cv'] = df_cv[[f'cv_{f}' for f in SHAPE_FEATS]].mean(axis=1)

pvc6_cv_rows = []
for cid_p in ['c1', 'c2']:
    df_p = pd.read_pickle(os.path.join(PVC6_PKL, f'df_all_{cid_p}.pkl'))
    row = {'cid': f'pvc6_{cid_p}', 'ct': 'pvc6',
           'r2_exp':  np.nanmedian(pvc6_r2[cid_p]['r2_exp']),
           'r2_ramp': np.nanmedian(pvc6_r2[cid_p]['r2_ramp'])}
    for feat in SHAPE_FEATS:
        if feat in df_p.columns:
            vals = df_p[feat].dropna().values
            mu = np.abs(np.nanmean(vals))
            row[f'cv_{feat}'] = np.nanstd(vals) / mu if mu > 1e-9 else np.nan
        else:
            row[f'cv_{feat}'] = np.nan
    pvc6_cv_rows.append(row)
df_cv_pvc6 = pd.DataFrame(pvc6_cv_rows)
df_cv_pvc6['mean_cv'] = df_cv_pvc6[[f'cv_{f}' for f in SHAPE_FEATS]].mean(axis=1)

_pc_cells = [d for d in spe1_r2 if d['cell_type'] == 'PC']
_in_cells = [d for d in spe1_r2 if d['cell_type'] == 'IN']


# ── figure ────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(28, 20))
gs = gridspec.GridSpec(3, 2, figure=fig,
                       hspace=0.58, wspace=0.38,
                       width_ratios=[1, 2.4],
                       height_ratios=[1, 1, 1])


# ── A: exp fit error (1-R²), log scale ───────────────────────────────────────
ax_a = fig.add_subplot(gs[0, 0])

_e_spe1_exp = np.clip(1 - _pool_spe1('r2_exp'), 1e-6, None)
_e_pvc6_exp = np.clip(1 - _pool_pvc6('r2_exp'), 1e-6, None)

_boxes(ax_a, [_e_spe1_exp, _e_pvc6_exp], [0, 1], [COL_SPE1, COL_PVC6], widths=0.65)
_fit_err_yax(ax_a)
ax_a.set_xlim(-0.65, 1.65)
ax_a.set_xticks([0, 1])
ax_a.set_xticklabels(['spe-1', 'pvc-6'], fontsize=_FS_TK, fontweight='bold')
ax_a.set_ylabel('Fit error  (1 − R²)', fontsize=_FS_AX, fontweight='bold')
ax_a.set_title('Exp. decay  fit', fontsize=_FS_AX, fontweight='bold', pad=10, loc='left')
_style(ax_a)


# ── B: exp R² per spe-1 cell (PC / IN coloured) ──────────────────────────────
ax_b = fig.add_subplot(gs[0, 1])

b_data, b_cols, b_labs, b_pos = [], [], [], []
_x = 0
for d in _pc_cells:
    v = d['r2_exp'][~np.isnan(d['r2_exp'])]
    b_data.append(v); b_cols.append(COL_PC)
    b_labs.append(d['cell_id']); b_pos.append(_x); _x += 1
_sep_b = _x - 0.5; _x += 0.7
for d in _in_cells:
    v = d['r2_exp'][~np.isnan(d['r2_exp'])]
    b_data.append(v); b_cols.append(COL_IN)
    b_labs.append(d['cell_id']); b_pos.append(_x); _x += 1

_boxes(ax_b, b_data, b_pos, b_cols)
ax_b.axvline(_sep_b, color='#dddddd', lw=1.5, ls='--')
_r2_yax(ax_b)
ax_b.set_xticks(b_pos)
ax_b.set_xticklabels(b_labs, rotation=65, ha='right', fontsize=_FS_SM, fontweight='bold')
ax_b.set_ylabel('R²', fontsize=_FS_AX, fontweight='bold')
ax_b.set_title('Exp. decay R²  –  per cell', fontsize=_FS_AX, fontweight='bold', pad=10, loc='left')
_style(ax_b, xtick_sz=_FS_SM)
ax_b.tick_params(axis='y', labelsize=_FS_TK)
for lbl in ax_b.get_yticklabels(): lbl.set_fontweight('bold')


# ── C: ramp fit error (1-R²), log scale ──────────────────────────────────────
ax_c = fig.add_subplot(gs[1, 0])

_e_spe1_ramp = np.clip(1 - _pool_spe1('r2_ramp'), 1e-6, None)
_e_pvc6_ramp = np.clip(1 - _pool_pvc6('r2_ramp'), 1e-6, None)

_boxes(ax_c, [_e_spe1_ramp, _e_pvc6_ramp], [0, 1], [COL_SPE1, COL_PVC6], widths=0.65)
_fit_err_yax(ax_c)
ax_c.set_xlim(-0.65, 1.65)
ax_c.set_xticks([0, 1])
ax_c.set_xticklabels(['spe-1', 'pvc-6'], fontsize=_FS_TK, fontweight='bold')
ax_c.set_ylabel('Fit error  (1 − R²)', fontsize=_FS_AX, fontweight='bold')
ax_c.set_title('Ramp  fit', fontsize=_FS_AX, fontweight='bold', pad=10, loc='left')
_style(ax_c)


# ── D: ramp R² per spe-1 cell ────────────────────────────────────────────────
ax_d = fig.add_subplot(gs[1, 1])

d_data, d_cols, d_labs, d_pos = [], [], [], []
_x = 0
for d in _pc_cells:
    v = d['r2_ramp'][~np.isnan(d['r2_ramp'])]
    d_data.append(v); d_cols.append(COL_PC)
    d_labs.append(d['cell_id']); d_pos.append(_x); _x += 1
_sep_d = _x - 0.5; _x += 0.7
for d in _in_cells:
    v = d['r2_ramp'][~np.isnan(d['r2_ramp'])]
    d_data.append(v); d_cols.append(COL_IN)
    d_labs.append(d['cell_id']); d_pos.append(_x); _x += 1

_boxes(ax_d, d_data, d_pos, d_cols)
ax_d.axvline(_sep_d, color='#dddddd', lw=1.5, ls='--')
_r2_yax(ax_d)
ax_d.set_xticks(d_pos)
ax_d.set_xticklabels(d_labs, rotation=65, ha='right', fontsize=_FS_SM, fontweight='bold')
ax_d.set_ylabel('R²', fontsize=_FS_AX, fontweight='bold')
ax_d.set_title('Ramp R²  –  per cell', fontsize=_FS_AX, fontweight='bold', pad=10, loc='left')
_style(ax_d, xtick_sz=_FS_SM)
ax_d.tick_params(axis='y', labelsize=_FS_TK)
for lbl in ax_d.get_yticklabels(): lbl.set_fontweight('bold')


# ── E: waveform feature variability (CV per feature across spe-1 cells) ───────
ax_e = fig.add_subplot(gs[2, 0])

e_data = [df_cv[f'cv_{f}'].dropna().values for f in SHAPE_FEATS]
_boxes(ax_e, e_data, list(range(len(SHAPE_FEATS))), [COL_SPE1] * len(SHAPE_FEATS), widths=0.6)

# overlay pvc-6 individual points
for fi, feat in enumerate(SHAPE_FEATS):
    for _, rp in df_cv_pvc6.iterrows():
        v = rp.get(f'cv_{feat}', np.nan)
        if not np.isnan(v):
            ax_e.scatter(fi, v, color=COL_PVC6, s=130, marker='D',
                        zorder=5, edgecolors='white', linewidths=0.8)

ax_e.set_xticks(range(len(SHAPE_FEATS)))
ax_e.set_xticklabels(FEAT_NICE, fontsize=_FS_TK, fontweight='bold')
ax_e.set_ylabel('CV  (σ / |μ|)', fontsize=_FS_AX, fontweight='bold')
ax_e.set_title('Feature variability', fontsize=_FS_AX, fontweight='bold', pad=10, loc='left')
ax_e.set_xlim(-0.6, len(SHAPE_FEATS) - 0.4)
_style(ax_e)


# ── F: mean CV vs exp R² — feature variability does not hurt fit quality ──────
ax_f = fig.add_subplot(gs[2, 1])

for ct, col in [('PC', COL_PC), ('IN', COL_IN)]:
    sub = df_cv[df_cv['ct'] == ct]
    ax_f.scatter(sub['mean_cv'], sub['r2_exp'], color=col, s=90,
                alpha=0.85, zorder=3, edgecolors='none')

for _, rp in df_cv_pvc6.iterrows():
    ax_f.scatter(rp['mean_cv'], rp['r2_exp'], color=COL_PVC6,
                s=200, marker='D', alpha=0.95, zorder=5,
                edgecolors='white', linewidths=0.8)

# regression (spe-1 only)
_xv = df_cv['mean_cv'].values
_yv = df_cv['r2_exp'].values
_m = ~(np.isnan(_xv) | np.isnan(_yv))
_r, _p = pearsonr(_xv[_m], _yv[_m])
_slope, _intercept = np.polyfit(_xv[_m], _yv[_m], 1)
_xx = np.linspace(_xv[_m].min(), _xv[_m].max(), 100)
ax_f.plot(_xx, _slope * _xx + _intercept, color='#999999', ls='--', lw=1.8, zorder=1, alpha=0.7)

_p_str = 'p < 0.001' if _p < 0.001 else f'p = {_p:.3f}'
ax_f.text(0.97, 0.05, f'r = {_r:.2f},  {_p_str}',
          transform=ax_f.transAxes, ha='right', va='bottom',
          fontsize=_FS_SM + 1, fontweight='bold', color='#444444')

ax_f.set_xlabel('Mean feature CV', fontsize=_FS_AX, fontweight='bold')
ax_f.set_ylabel('Median exp. R²', fontsize=_FS_AX, fontweight='bold')
ax_f.set_title('Feature variability vs. fit quality', fontsize=_FS_AX, fontweight='bold', pad=10, loc='left')
_style(ax_f)
ax_f.tick_params(axis='both', labelsize=_FS_TK)
for lbl in ax_f.get_xticklabels() + ax_f.get_yticklabels(): lbl.set_fontweight('bold')


# ── shared legend ─────────────────────────────────────────────────────────────
fig.legend(
    handles=[mpatches.Patch(facecolor=COL_SPE1, alpha=_ALPHA, label='spe-1  (all)'),
             mpatches.Patch(facecolor=COL_PC,   alpha=_ALPHA, label='spe-1  PC'),
             mpatches.Patch(facecolor=COL_IN,   alpha=_ALPHA, label='spe-1  IN'),
             mpatches.Patch(facecolor=COL_PVC6, alpha=_ALPHA, label='pvc-6')],
    loc='upper right', frameon=False,
    prop={'size': _FS_AX - 2, 'weight': 'bold'},
    bbox_to_anchor=(0.995, 1.0))

plt.show()